In [11]:
project_root = Path.cwd().parent
raw_data_path = project_root / "data" / "raw"

csv_files = sorted(raw_data_path.glob("*.csv"))

for file in csv_files:
    print(file.name)

#print(f"\nTotal CSV files: {len(csv_files)}")


campaign_desc.csv
campaign_table.csv
causal_data.csv
coupon.csv
coupon_redempt.csv
hh_demographic.csv
product.csv
transaction_data.csv


In [12]:
import pandas as pd

campaign_desc = pd.read_csv(raw_data_path / "campaign_desc.csv")

print(f"Rows: {campaign_desc.shape[0]}")
print(f"Columns: {campaign_desc.shape[1]}")

campaign_desc.head()

Rows: 30
Columns: 4


,DESCRIPTION,CAMPAIGN,START_DAY,END_DAY
0,TypeB,24,659,719
1,TypeC,15,547,708
2,TypeB,25,659,691
3,TypeC,20,615,685
4,TypeB,23,646,684


In [13]:
campaign_desc.info()

<class 'pandas.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   DESCRIPTION  30 non-null     str  
 1   CAMPAIGN     30 non-null     int64
 2   START_DAY    30 non-null     int64
 3   END_DAY      30 non-null     int64
dtypes: int64(3), str(1)
memory usage: 1.2 KB


In [14]:
duplicate_campaigns = campaign_desc["CAMPAIGN"].duplicated().sum()

print(f"Duplicate campaign IDs: {duplicate_campaigns}")

Duplicate campaign IDs: 0


In [15]:
invalid_dates = campaign_desc["END_DAY"] < campaign_desc["START_DAY"]

print(f"Campaigns ending before they start: {invalid_dates.sum()}")

Campaigns ending before they start: 0


In [16]:
campaign_desc["DESCRIPTION"].value_counts()

DESCRIPTION
TypeB    19
TypeC     6
TypeA     5
Name: count, dtype: int64

In [17]:
campaign_table = pd.read_csv(raw_data_path / "campaign_table.csv")

print(f"Rows: {campaign_table.shape[0]}")
print(f"Columns: {campaign_table.shape[1]}")

campaign_table.head()

Rows: 7208
Columns: 3


,DESCRIPTION,household_key,CAMPAIGN
0,TypeA,17,26
1,TypeA,27,26
2,TypeA,212,26
3,TypeA,208,26
4,TypeA,192,26


In [18]:
campaign_table.info()

<class 'pandas.DataFrame'>
RangeIndex: 7208 entries, 0 to 7207
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   DESCRIPTION    7208 non-null   str  
 1   household_key  7208 non-null   int64
 2   CAMPAIGN       7208 non-null   int64
dtypes: int64(2), str(1)
memory usage: 204.3 KB


In [19]:
duplicate_pairs = campaign_table.duplicated(
    subset=["household_key", "CAMPAIGN"]
).sum()

print(f"Duplicate household–campaign pairs: {duplicate_pairs}")

Duplicate household–campaign pairs: 0


In [20]:
unknown_campaigns = campaign_table.loc[
    ~campaign_table["CAMPAIGN"].isin(campaign_desc["CAMPAIGN"]),
    "CAMPAIGN"
].unique()

print(f"Campaign IDs missing from campaign_desc: {unknown_campaigns}")

Campaign IDs missing from campaign_desc: []


In [21]:
targeted_households = campaign_table["household_key"].nunique()

print(f"Unique households targeted: {targeted_households}")

Unique households targeted: 1584


In [22]:
households_per_campaign = (
    campaign_table.groupby("CAMPAIGN")["household_key"]
    .nunique()
    .sort_values(ascending=False)
)

households_per_campaign

CAMPAIGN
18    1133
13    1077
8     1076
30     361
26     332
22     276
20     244
14     224
11     214
17     202
7      198
16     188
25     187
23     183
9      176
12     170
5      166
19     130
10     123
29     118
24     100
4       81
21      65
6       65
2       48
28      17
15      17
1       13
3       12
27      12
Name: household_key, dtype: int64

In [26]:
for campaign_id, household_count in households_per_campaign.iloc[-6:].items():
    print(f"Campaign {campaign_id}: {household_count} households")

Campaign 2: 48 households
Campaign 28: 17 households
Campaign 15: 17 households
Campaign 1: 13 households
Campaign 3: 12 households
Campaign 27: 12 households


In [27]:
campaign_summary = campaign_desc.merge(
    households_per_campaign.rename("targeted_households"),
    left_on="CAMPAIGN",
    right_index=True,
    how="left",
    validate="one_to_one"
)

campaign_summary.head()

,DESCRIPTION,CAMPAIGN,START_DAY,END_DAY,targeted_households
0,TypeB,24,659,719,100
1,TypeC,15,547,708,17
2,TypeB,25,659,691,187
3,TypeC,20,615,685,244
4,TypeB,23,646,684,183


In [28]:
missing_audience_counts = campaign_summary["targeted_households"].isna().sum()

print(f"Campaigns missing an audience count: {missing_audience_counts}")

Campaigns missing an audience count: 0


In [29]:
campaign_summary["duration_days"] = (
    campaign_summary["END_DAY"] - campaign_summary["START_DAY"] + 1
)

campaign_summary[
    ["CAMPAIGN", "START_DAY", "END_DAY", "duration_days"]
].head()

,CAMPAIGN,START_DAY,END_DAY,duration_days
0,24,659,719,61
1,15,547,708,162
2,25,659,691,33
3,20,615,685,71
4,23,646,684,39


In [30]:
campaign_summary[["duration_days", "targeted_households"]].describe()

,duration_days,targeted_households
count,30.000000,30.000000
mean,47.600000,240.266667
std,24.856277,304.411579
min,33.000000,12.000000
25%,33.000000,69.000000
50%,38.500000,173.000000
75%,55.500000,221.500000
max,162.000000,1133.000000


In [31]:
campaign_type_summary = campaign_summary.groupby("DESCRIPTION").agg(
    campaign_count=("CAMPAIGN", "count"),
    median_duration_days=("duration_days", "median"),
    median_targeted_households=("targeted_households", "median")
)

campaign_type_summary

,campaign_count,median_duration_days,median_targeted_households
DESCRIPTION,,,
TypeA,5,48.0,1076.0
TypeB,19,33.0,166.0
TypeC,6,65.0,41.0


In [32]:
campaign_type_check = campaign_table.merge(
    campaign_desc[["CAMPAIGN", "DESCRIPTION"]],
    on="CAMPAIGN",
    suffixes=("_targeting", "_description"),
    validate="many_to_one"
)

type_mismatches = (
    campaign_type_check["DESCRIPTION_targeting"]
    != campaign_type_check["DESCRIPTION_description"]
).sum()

print(f"Campaign-type mismatches: {type_mismatches}")

Campaign-type mismatches: 0


In [33]:
coupon = pd.read_csv(raw_data_path / "coupon.csv")

print(f"Rows: {coupon.shape[0]}")
print(f"Columns: {coupon.shape[1]}")

coupon.head()

Rows: 124548
Columns: 3


,COUPON_UPC,PRODUCT_ID,CAMPAIGN
0,10000089061,27160,4
1,10000089064,27754,9
2,10000089073,28897,12
3,51800009050,28919,28
4,52100000076,28929,25


In [34]:
coupon.info()

<class 'pandas.DataFrame'>
RangeIndex: 124548 entries, 0 to 124547
Data columns (total 3 columns):
 #   Column      Non-Null Count   Dtype
---  ------      --------------   -----
 0   COUPON_UPC  124548 non-null  int64
 1   PRODUCT_ID  124548 non-null  int64
 2   CAMPAIGN    124548 non-null  int64
dtypes: int64(3)
memory usage: 2.9 MB


In [35]:
duplicate_coupon_rows = coupon.duplicated(
    subset=["COUPON_UPC", "PRODUCT_ID", "CAMPAIGN"]
).sum()

print(f"Duplicate coupon–product–campaign combinations: {duplicate_coupon_rows}")

Duplicate coupon–product–campaign combinations: 5164


In [36]:
duplicate_coupon_examples = coupon[
    coupon.duplicated(
        subset=["COUPON_UPC", "PRODUCT_ID", "CAMPAIGN"],
        keep=False
    )
].sort_values(["COUPON_UPC", "PRODUCT_ID", "CAMPAIGN"])

duplicate_coupon_examples.head(10)

,COUPON_UPC,PRODUCT_ID,CAMPAIGN
73199,10000085426,930799,13
73748,10000085426,930799,13
62387,10000085427,983002,13
64785,10000085427,983002,13
64507,10000085427,1078612,13
64908,10000085427,1078612,13
64913,10000085427,13072690,13
64933,10000085427,13072690,13
59992,10000085427,13911346,13
64343,10000085427,13911346,13


In [37]:
coupon_clean = coupon.drop_duplicates(
    subset=["COUPON_UPC", "PRODUCT_ID", "CAMPAIGN"]
).copy()

print(f"Original rows: {len(coupon)}")
print(f"Cleaned rows: {len(coupon_clean)}")
print(f"Rows removed: {len(coupon) - len(coupon_clean)}")

Original rows: 124548
Cleaned rows: 119384
Rows removed: 5164


In [38]:
unknown_coupon_campaigns = coupon_clean.loc[
    ~coupon_clean["CAMPAIGN"].isin(campaign_desc["CAMPAIGN"]),
    "CAMPAIGN"
].unique()

print(f"Coupon campaign IDs missing from campaign_desc: {unknown_coupon_campaigns}")

Coupon campaign IDs missing from campaign_desc: []


In [39]:
unique_coupons = coupon_clean["COUPON_UPC"].nunique()

print(f"Unique coupon identifiers: {unique_coupons}")

Unique coupon identifiers: 1135


In [40]:
campaigns_per_coupon = coupon_clean.groupby("COUPON_UPC")["CAMPAIGN"].nunique()

print(
    f"Coupon identifiers used in multiple campaigns: "
    f"{(campaigns_per_coupon > 1).sum()}"
)

Coupon identifiers used in multiple campaigns: 171


In [41]:
coupon_redempt = pd.read_csv(raw_data_path / "coupon_redempt.csv")

print(f"Rows: {coupon_redempt.shape[0]}")
print(f"Columns: {coupon_redempt.shape[1]}")

coupon_redempt.head()

Rows: 2318
Columns: 4


,household_key,DAY,COUPON_UPC,CAMPAIGN
0,1,421,10000085364,8
1,1,421,51700010076,8
2,1,427,54200000033,8
3,1,597,10000085476,18
4,1,597,54200029176,18


In [42]:
coupon_redempt.info()

<class 'pandas.DataFrame'>
RangeIndex: 2318 entries, 0 to 2317
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   household_key  2318 non-null   int64
 1   DAY            2318 non-null   int64
 2   COUPON_UPC     2318 non-null   int64
 3   CAMPAIGN       2318 non-null   int64
dtypes: int64(4)
memory usage: 72.6 KB


In [43]:
duplicate_redemptions = coupon_redempt.duplicated(
    subset=["household_key", "DAY", "COUPON_UPC", "CAMPAIGN"]
).sum()

print(f"Repeated redemption rows: {duplicate_redemptions}")

Repeated redemption rows: 0


In [44]:
valid_coupon_campaigns = coupon_clean[
    ["COUPON_UPC", "CAMPAIGN"]
].drop_duplicates()

redemption_check = coupon_redempt.merge(
    valid_coupon_campaigns,
    on=["COUPON_UPC", "CAMPAIGN"],
    how="left",
    indicator=True,
    validate="many_to_one"
)

unmatched_redemptions = (redemption_check["_merge"] == "left_only").sum()

print(f"Redemption rows without a matching coupon–campaign pair: {unmatched_redemptions}")

Redemption rows without a matching coupon–campaign pair: 0


In [45]:
redemption_targeting_check = coupon_redempt.merge(
    campaign_table[["household_key", "CAMPAIGN"]],
    on=["household_key", "CAMPAIGN"],
    how="left",
    indicator=True,
    validate="many_to_one"
)

unmatched_targeting = (
    redemption_targeting_check["_merge"] == "left_only"
).sum()

print(f"Redemption rows without a matching targeting record: {unmatched_targeting}")

Redemption rows without a matching targeting record: 0


In [46]:
redemption_dates = coupon_redempt.merge(
    campaign_desc[["CAMPAIGN", "START_DAY", "END_DAY"]],
    on="CAMPAIGN",
    how="left",
    validate="many_to_one"
)

outside_campaign = (
    (redemption_dates["DAY"] < redemption_dates["START_DAY"])
    | (redemption_dates["DAY"] > redemption_dates["END_DAY"])
)

print(f"Redemption rows outside campaign dates: {outside_campaign.sum()}")

Redemption rows outside campaign dates: 0


In [47]:
redeeming_households = coupon_redempt["household_key"].nunique()

print(f"Unique households with a recorded redemption: {redeeming_households}")

Unique households with a recorded redemption: 434


In [48]:
household_redemption_rate = (
    redeeming_households / targeted_households * 100
)

print(
    f"Targeted households with a recorded redemption: "
    f"{household_redemption_rate:.2f}%"
)

Targeted households with a recorded redemption: 27.40%


In [49]:
redeemers_per_campaign = (
    coupon_redempt.groupby("CAMPAIGN")["household_key"]
    .nunique()
    .rename("redeeming_households")
)

redeemers_per_campaign.head()

CAMPAIGN
1    1
2    2
3    2
4    6
5    8
Name: redeeming_households, dtype: int64

In [50]:
campaign_summary = campaign_summary.join(
    redeemers_per_campaign,
    on="CAMPAIGN",
    validate="one_to_one"
)

campaign_summary["redeeming_households"] = (
    campaign_summary["redeeming_households"].fillna(0).astype(int)
)

campaign_summary[
    ["CAMPAIGN", "targeted_households", "redeeming_households"]
].head()

,CAMPAIGN,targeted_households,redeeming_households
0,24,100,7
1,15,17,2
2,25,187,24
3,20,244,20
4,23,183,23


In [51]:
campaign_summary["household_redemption_pct"] = (
    campaign_summary["redeeming_households"]
    / campaign_summary["targeted_households"]
    * 100
)

campaign_summary[
    [
        "CAMPAIGN",
        "targeted_households",
        "redeeming_households",
        "household_redemption_pct"
    ]
].head().round(2)

,CAMPAIGN,targeted_households,redeeming_households,household_redemption_pct
0,24,100,7,7.00
1,15,17,2,11.76
2,25,187,24,12.83
3,20,244,20,8.20
4,23,183,23,12.57


In [52]:
campaign_summary.sort_values(
    "household_redemption_pct",
    ascending=False
)[
    [
        "CAMPAIGN",
        "DESCRIPTION",
        "targeted_households",
        "redeeming_households",
        "household_redemption_pct"
    ]
].head(5).round(2)

,CAMPAIGN,DESCRIPTION,targeted_households,redeeming_households,household_redemption_pct
7,18,TypeA,1133,214,18.89
12,13,TypeA,1077,196,18.20
20,3,TypeC,12,2,16.67
17,8,TypeA,1076,158,14.68
2,25,TypeB,187,24,12.83


In [53]:
zero_redemption_campaigns = campaign_summary.loc[
    campaign_summary["redeeming_households"] == 0,
    ["CAMPAIGN", "DESCRIPTION", "targeted_households"]
]

print(f"Campaigns with no recorded redemptions: {len(zero_redemption_campaigns)}")
zero_redemption_campaigns

Campaigns with no recorded redemptions: 0


,CAMPAIGN,DESCRIPTION,targeted_households


In [54]:
product = pd.read_csv(raw_data_path / "product.csv")

print(f"Rows: {product.shape[0]}")
print(f"Columns: {product.shape[1]}")

product.head()

Rows: 92353
Columns: 7


,PRODUCT_ID,MANUFACTURER,DEPARTMENT,BRAND,COMMODITY_DESC,SUB_COMMODITY_DESC,CURR_SIZE_OF_PRODUCT
0,25671,2,GROCERY,National,FRZN ICE,ICE - CRUSHED/CUBED,22 LB
1,26081,2,MISC. TRANS.,National,NO COMMODITY DESCRIPTION,NO SUBCOMMODITY DESCRIPTION,
2,26093,69,PASTRY,Private,BREAD,BREAD:ITALIAN/FRENCH,
3,26190,69,GROCERY,Private,FRUIT - SHELF STABLE,APPLE SAUCE,50 OZ
4,26355,69,GROCERY,Private,COOKIES/CONES,SPECIALTY COOKIES,14 OZ


In [55]:
product.info()

<class 'pandas.DataFrame'>
RangeIndex: 92353 entries, 0 to 92352
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   PRODUCT_ID            92353 non-null  int64
 1   MANUFACTURER          92353 non-null  int64
 2   DEPARTMENT            92353 non-null  str  
 3   BRAND                 92353 non-null  str  
 4   COMMODITY_DESC        92353 non-null  str  
 5   SUB_COMMODITY_DESC    92353 non-null  str  
 6   CURR_SIZE_OF_PRODUCT  92353 non-null  str  
dtypes: int64(2), str(5)
memory usage: 9.5 MB


In [56]:
text_columns = [
    "DEPARTMENT",
    "BRAND",
    "COMMODITY_DESC",
    "SUB_COMMODITY_DESC",
    "CURR_SIZE_OF_PRODUCT"
]

blank_counts = product[text_columns].apply(
    lambda column: column.str.strip().eq("").sum()
)

print(blank_counts)

DEPARTMENT                 15
BRAND                       0
COMMODITY_DESC             15
SUB_COMMODITY_DESC         15
CURR_SIZE_OF_PRODUCT    30607
dtype: int64


In [57]:
duplicate_product_ids = product["PRODUCT_ID"].duplicated().sum()

print(f"Duplicate product IDs: {duplicate_product_ids}")

Duplicate product IDs: 0


In [58]:
unknown_coupon_products = coupon_clean.loc[
    ~coupon_clean["PRODUCT_ID"].isin(product["PRODUCT_ID"]),
    "PRODUCT_ID"
].unique()

print(f"Distinct coupon product IDs missing from product: {len(unknown_coupon_products)}")

Distinct coupon product IDs missing from product: 0


In [59]:
product_clean = product.copy()

for column in text_columns:
    product_clean[column] = (
        product_clean[column]
        .str.strip()
        .replace("", pd.NA)
    )

product_clean[text_columns].isna().sum()

DEPARTMENT                 15
BRAND                       0
COMMODITY_DESC             15
SUB_COMMODITY_DESC         15
CURR_SIZE_OF_PRODUCT    30607
dtype: int64

In [60]:
placeholder_counts = pd.Series({
    "COMMODITY_DESC": product_clean["COMMODITY_DESC"]
        .eq("NO COMMODITY DESCRIPTION").sum(),
    "SUB_COMMODITY_DESC": product_clean["SUB_COMMODITY_DESC"]
        .eq("NO SUBCOMMODITY DESCRIPTION").sum()
})

print(placeholder_counts)

COMMODITY_DESC        490
SUB_COMMODITY_DESC    429
dtype: int64


In [61]:
product_clean["COMMODITY_DESC"] = product_clean["COMMODITY_DESC"].replace(
    "NO COMMODITY DESCRIPTION", pd.NA
)

product_clean["SUB_COMMODITY_DESC"] = product_clean["SUB_COMMODITY_DESC"].replace(
    "NO SUBCOMMODITY DESCRIPTION", pd.NA
)

product_clean[["COMMODITY_DESC", "SUB_COMMODITY_DESC"]].isna().sum()

COMMODITY_DESC        505
SUB_COMMODITY_DESC    444
dtype: int64

In [62]:
product_clean["DEPARTMENT"].value_counts(dropna=False).head(10)

DEPARTMENT
GROCERY       39021
DRUG GM       31529
PRODUCE        3118
COSMETICS      3011
NUTRITION      2914
MEAT           2544
MEAT-PCKGD     2427
DELI           2354
PASTRY         2149
FLORAL          938
Name: count, dtype: int64

In [63]:
hh_demographic = pd.read_csv(raw_data_path / "hh_demographic.csv")

print(f"Rows: {hh_demographic.shape[0]}")
print(f"Columns: {hh_demographic.shape[1]}")

hh_demographic.head()

Rows: 801
Columns: 8


,classification_1,classification_2,classification_3,HOMEOWNER_DESC,classification_5,classification_4,KID_CATEGORY_DESC,household_key
0,Age Group6,X,Level4,Homeowner,Group5,2,None/Unknown,1
1,Age Group4,X,Level5,Homeowner,Group5,2,None/Unknown,7
2,Age Group2,Y,Level3,Unknown,Group4,3,1,8
3,Age Group2,Y,Level6,Homeowner,Group4,4,2,13
4,Age Group4,Z,Level5,Homeowner,Group3,1,None/Unknown,16


In [64]:
hh_demographic.info()

<class 'pandas.DataFrame'>
RangeIndex: 801 entries, 0 to 800
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   classification_1   801 non-null    str  
 1   classification_2   801 non-null    str  
 2   classification_3   801 non-null    str  
 3   HOMEOWNER_DESC     801 non-null    str  
 4   classification_5   801 non-null    str  
 5   classification_4   801 non-null    str  
 6   KID_CATEGORY_DESC  801 non-null    str  
 7   household_key      801 non-null    int64
dtypes: int64(1), str(7)
memory usage: 82.5 KB


In [65]:
duplicate_demographic_households = (
    hh_demographic["household_key"].duplicated().sum()
)

print(f"Duplicate household IDs: {duplicate_demographic_households}")

Duplicate household IDs: 0


In [66]:
targeted_household_ids = campaign_table["household_key"].drop_duplicates()

has_demographics = targeted_household_ids.isin(
    hh_demographic["household_key"]
)

print(f"Targeted households with demographics: {has_demographics.sum()}")
print(f"Targeted households without demographics: {(~has_demographics).sum()}")

Targeted households with demographics: 760
Targeted households without demographics: 824


In [67]:
demographic_columns = hh_demographic.columns.drop("household_key")

demographic_blank_counts = hh_demographic[demographic_columns].apply(
    lambda column: column.str.strip().eq("").sum()
)

print(demographic_blank_counts)

classification_1     0
classification_2     0
classification_3     0
HOMEOWNER_DESC       0
classification_5     0
classification_4     0
KID_CATEGORY_DESC    0
dtype: int64


In [68]:
hh_demographic["HOMEOWNER_DESC"].value_counts(dropna=False)

HOMEOWNER_DESC
Homeowner          504
Unknown            233
Renter              42
Probable Renter     11
Probable Owner      11
Name: count, dtype: int64

In [69]:
hh_demographic["KID_CATEGORY_DESC"].value_counts(dropna=False)

KID_CATEGORY_DESC
None/Unknown    558
1               114
3+               69
2                60
Name: count, dtype: int64

In [70]:
hh_demographic["classification_1"].value_counts(dropna=False)

classification_1
Age Group4    288
Age Group3    194
Age Group2    142
Age Group6     72
Age Group5     59
Age Group1     46
Name: count, dtype: int64

In [71]:
hh_demographic["classification_2"].value_counts(dropna=False)

classification_2
Y    344
X    340
Z    117
Name: count, dtype: int64

In [72]:
hh_demographic["classification_3"].value_counts(dropna=False)

classification_3
Level5     192
Level4     172
Level6      96
Level3      77
Level2      74
Level1      61
Level8      38
Level7      34
Level9      30
Level12     11
Level10     11
Level11      5
Name: count, dtype: int64

In [73]:
hh_demographic["classification_4"].value_counts(dropna=False)

classification_4
2     318
1     255
3     109
5+     66
4      53
Name: count, dtype: int64

In [74]:
hh_demographic["classification_5"].value_counts(dropna=False)

classification_5
Group5    255
Group4    187
Group3    144
Group2     95
Group6     73
Group1     47
Name: count, dtype: int64

In [75]:
transaction_preview = pd.read_csv(
    raw_data_path / "transaction_data.csv",
    nrows=5
)

print(f"Columns: {transaction_preview.shape[1]}")
transaction_preview

Columns: 12


,household_key,BASKET_ID,DAY,PRODUCT_ID,QUANTITY,SALES_VALUE,STORE_ID,RETAIL_DISC,TRANS_TIME,WEEK_NO,COUPON_DISC,COUPON_MATCH_DISC
0,2375,26984851472,1,1004906,1,1.39,364,-0.60,1631,1,0,0
1,2375,26984851472,1,1033142,1,0.82,364,0.00,1631,1,0,0
2,2375,26984851472,1,1036325,1,0.99,364,-0.30,1631,1,0,0
3,2375,26984851472,1,1082185,1,1.21,364,0.00,1631,1,0,0
4,2375,26984851472,1,8160430,1,1.50,364,-0.39,1631,1,0,0


In [76]:
transaction_data = pd.read_csv(
    raw_data_path / "transaction_data.csv"
)

transaction_data.info(show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 2595732 entries, 0 to 2595731
Data columns (total 12 columns):
 #   Column             Non-Null Count    Dtype  
---  ------             --------------    -----  
 0   household_key      2595732 non-null  int64  
 1   BASKET_ID          2595732 non-null  int64  
 2   DAY                2595732 non-null  int64  
 3   PRODUCT_ID         2595732 non-null  int64  
 4   QUANTITY           2595732 non-null  int64  
 5   SALES_VALUE        2595732 non-null  float64
 6   STORE_ID           2595732 non-null  int64  
 7   RETAIL_DISC        2595732 non-null  float64
 8   TRANS_TIME         2595732 non-null  int64  
 9   WEEK_NO            2595732 non-null  int64  
 10  COUPON_DISC        2595732 non-null  float64
 11  COUPON_MATCH_DISC  2595732 non-null  float64
dtypes: float64(4), int64(8)
memory usage: 237.6 MB


In [77]:
print(f"Unique households: {transaction_data['household_key'].nunique():,}")
print(f"Unique baskets: {transaction_data['BASKET_ID'].nunique():,}")
print(f"Unique products: {transaction_data['PRODUCT_ID'].nunique():,}")

Unique households: 2,500
Unique baskets: 276,484
Unique products: 92,339


In [78]:
unknown_transaction_products = transaction_data.loc[
    ~transaction_data["PRODUCT_ID"].isin(product_clean["PRODUCT_ID"]),
    "PRODUCT_ID"
].unique()

print(
    f"Distinct transaction product IDs missing from the catalog: "
    f"{len(unknown_transaction_products)}"
)

Distinct transaction product IDs missing from the catalog: 0


In [79]:
print(f"First recorded day: {transaction_data['DAY'].min()}")
print(f"Last recorded day: {transaction_data['DAY'].max()}")
print(f"First recorded week: {transaction_data['WEEK_NO'].min()}")
print(f"Last recorded week: {transaction_data['WEEK_NO'].max()}")

First recorded day: 1
Last recorded day: 711
First recorded week: 1
Last recorded week: 102


In [80]:
last_transaction_day = transaction_data["DAY"].max()

campaigns_beyond_coverage = campaign_summary.loc[
    campaign_summary["END_DAY"] > last_transaction_day,
    ["CAMPAIGN", "START_DAY", "END_DAY"]
]

print(f"Campaigns ending after transaction coverage: {len(campaigns_beyond_coverage)}")
campaigns_beyond_coverage

Campaigns ending after transaction coverage: 1


,CAMPAIGN,START_DAY,END_DAY
0,24,659,719


In [81]:
duplicate_transaction_rows = transaction_data.duplicated().sum()

print(f"Repeated transaction rows: {duplicate_transaction_rows}")

Repeated transaction rows: 0


In [82]:
print(f"Rows with zero quantity: {(transaction_data['QUANTITY'] == 0).sum():,}")
print(f"Rows with negative quantity: {(transaction_data['QUANTITY'] < 0).sum():,}")
print(f"Rows with zero sales value: {(transaction_data['SALES_VALUE'] == 0).sum():,}")
print(f"Rows with negative sales value: {(transaction_data['SALES_VALUE'] < 0).sum():,}")

Rows with zero quantity: 14,466
Rows with negative quantity: 0
Rows with zero sales value: 18,850
Rows with negative sales value: 0


In [83]:
zero_value_summary = pd.crosstab(
    transaction_data["QUANTITY"].eq(0),
    transaction_data["SALES_VALUE"].eq(0),
    rownames=["Quantity is zero"],
    colnames=["Sales value is zero"]
)

zero_value_summary

Sales value is zero,False,True
Quantity is zero,,
False,2576815,4451
True,67,14399


In [84]:
zero_quantity_positive_sales = transaction_data.loc[
    (transaction_data["QUANTITY"] == 0)
    & (transaction_data["SALES_VALUE"] > 0)
]

zero_quantity_positive_sales.head()

,household_key,BASKET_ID,DAY,PRODUCT_ID,QUANTITY,SALES_VALUE,STORE_ID,RETAIL_DISC,TRANS_TIME,WEEK_NO,COUPON_DISC,COUPON_MATCH_DISC
6032,2263,27130336063,13,986912,0,7.000000e-02,31401,0.0,1717,3,0.0,0.0
136410,435,28085360560,91,2614183,0,4.000000e-02,3333,0.0,1349,14,0.0,0.0
211149,454,28344946496,114,914513,0,5.551115e-17,295,0.0,1849,17,0.0,0.0
211156,454,28344946496,114,973673,0,5.551115e-17,295,0.0,1849,17,0.0,0.0
304499,932,28757625745,138,858732,0,6.661338e-16,300,0.0,1400,20,0.0,0.0


In [85]:
import numpy as np

near_zero_sales = np.isclose(
    zero_quantity_positive_sales["SALES_VALUE"],
    0,
    atol=1e-8,
    rtol=0
)

print(f"Zero-quantity rows with near-zero positive sales: {near_zero_sales.sum()}")
print(f"Zero-quantity rows with sales above that tolerance: {(~near_zero_sales).sum()}")

Zero-quantity rows with near-zero positive sales: 29
Zero-quantity rows with sales above that tolerance: 38


In [86]:
zero_quantity_review = zero_quantity_positive_sales.loc[
    ~near_zero_sales
].merge(
    product_clean[
        ["PRODUCT_ID", "DEPARTMENT", "COMMODITY_DESC"]
    ],
    on="PRODUCT_ID",
    how="left",
    validate="many_to_one"
)

zero_quantity_review[
    ["PRODUCT_ID", "QUANTITY", "SALES_VALUE", "DEPARTMENT", "COMMODITY_DESC"]
].head(10)

,PRODUCT_ID,QUANTITY,SALES_VALUE,DEPARTMENT,COMMODITY_DESC
0,986912,0,0.07,DELI,PREPARED FOOD
1,2614183,0,0.04,PRODUCE,TOMATOES
2,839147,0,0.01,DRUG GM,CANDY - CHECKLANE
3,57059,0,0.01,PRODUCE,POTATOES
4,866871,0,5.82,PRODUCE,GRAPES
5,873026,0,0.01,GROCERY,PWDR/CRYSTL DRNK MX
6,956995,0,0.01,DRUG GM,CANDY - CHECKLANE
7,866858,0,0.50,DELI,DELI MEATS
8,5995423,0,0.01,DRUG GM,CANDY - CHECKLANE
9,990941,0,0.01,DRUG GM,CANDY - CHECKLANE


In [87]:
print(f"Flagged rows: {len(zero_quantity_review)}")
print(
    f"Total recorded sales value: "
    f"{zero_quantity_review['SALES_VALUE'].sum():.2f}"
)

Flagged rows: 38
Total recorded sales value: 19.28


In [88]:
zero_sales_review = transaction_data.loc[
    (transaction_data["QUANTITY"] > 0)
    & (transaction_data["SALES_VALUE"] == 0),
    [
        "PRODUCT_ID",
        "QUANTITY",
        "SALES_VALUE",
        "RETAIL_DISC",
        "COUPON_DISC",
        "COUPON_MATCH_DISC"
    ]
]

zero_sales_review.head()

,PRODUCT_ID,QUANTITY,SALES_VALUE,RETAIL_DISC,COUPON_DISC,COUPON_MATCH_DISC
766,837963,1,0.0,0.00,0.0,0.0
996,951412,1,0.0,-1.48,0.0,0.0
1095,850115,1,0.0,-3.29,0.0,0.0
1113,976872,1,0.0,-2.49,0.0,0.0
1161,940108,1,0.0,-1.19,0.0,0.0


In [89]:
discount_columns = [
    "RETAIL_DISC",
    "COUPON_DISC",
    "COUPON_MATCH_DISC"
]

has_discount_entry = zero_sales_review[discount_columns].ne(0).any(axis=1)

print(f"Rows with a nonzero discount entry: {has_discount_entry.sum():,}")
print(f"Rows with all discount entries zero: {(~has_discount_entry).sum():,}")

Rows with a nonzero discount entry: 3,749
Rows with all discount entries zero: 702


In [90]:
zero_sales_no_discount = zero_sales_review.loc[
    ~has_discount_entry
].merge(
    product_clean[["PRODUCT_ID", "DEPARTMENT", "COMMODITY_DESC"]],
    on="PRODUCT_ID",
    how="left",
    validate="many_to_one"
)

zero_sales_no_discount["COMMODITY_DESC"].value_counts(
    dropna=False
).head(10)

COMMODITY_DESC
COUPONS/STORE & MFG               444
BAG SNACKS                        225
COUPON/MISC ITEMS                   9
ROLLS                               4
TROPICAL FRUIT                      2
VITAMINS                            2
BAKED SWEET GOODS                   2
ANALGESICS                          1
GREETING CARDS/WRAP/PARTY SPLY      1
GARDEN CENTER                       1
Name: count, dtype: int64

In [91]:
zero_sales_no_discount.loc[
    zero_sales_no_discount["COMMODITY_DESC"] == "BAG SNACKS",
    "PRODUCT_ID"
].value_counts().head(10)

PRODUCT_ID
897752     54
1079228    46
991268     38
885430     20
1049131    16
837963     11
876660      6
1077422     6
943030      5
1030251     5
Name: count, dtype: int64

In [92]:
households_per_basket = (
    transaction_data.groupby("BASKET_ID")["household_key"].nunique()
)

print(
    f"Basket IDs linked to multiple households: "
    f"{(households_per_basket > 1).sum()}"
)

Basket IDs linked to multiple households: 0


In [93]:
basket_consistency = transaction_data.groupby("BASKET_ID").agg(
    distinct_days=("DAY", "nunique"),
    distinct_stores=("STORE_ID", "nunique")
)

print(f"Baskets spanning multiple days: {(basket_consistency['distinct_days'] > 1).sum()}")
print(f"Baskets spanning multiple stores: {(basket_consistency['distinct_stores'] > 1).sum()}")

Baskets spanning multiple days: 0
Baskets spanning multiple stores: 0


In [94]:
targeted_without_transactions = targeted_household_ids[
    ~targeted_household_ids.isin(transaction_data["household_key"])
]

print(
    f"Targeted households without transaction records: "
    f"{len(targeted_without_transactions)}"
)

Targeted households without transaction records: 0


In [95]:
causal_preview = pd.read_csv(
    raw_data_path / "causal_data.csv",
    nrows=5
)

print(f"Columns: {causal_preview.shape[1]}")
causal_preview

Columns: 5


,PRODUCT_ID,STORE_ID,WEEK_NO,display,mailer
0,26190,286,70,0,A
1,26190,288,70,0,A
2,26190,289,70,0,A
3,26190,292,70,0,A
4,26190,293,70,0,A


In [96]:
causal_row_count = 0

for chunk in pd.read_csv(
    raw_data_path / "causal_data.csv",
    usecols=["PRODUCT_ID"],
    chunksize=500_000
):
    causal_row_count += len(chunk)

print(f"Total promotion records: {causal_row_count:,}")

Total promotion records: 36,786,524


In [97]:
causal_missing_counts = pd.Series(
    0,
    index=causal_preview.columns,
    dtype="int64"
)

for chunk in pd.read_csv(
    raw_data_path / "causal_data.csv",
    dtype={"display": "string", "mailer": "string"},
    chunksize=500_000
):
    causal_missing_counts += chunk.isna().sum()

print(causal_missing_counts)

PRODUCT_ID    0
STORE_ID      0
WEEK_NO       0
display       0
mailer        0
dtype: int64


In [98]:
display_codes = set()
mailer_codes = set()

for chunk in pd.read_csv(
    raw_data_path / "causal_data.csv",
    usecols=["display", "mailer"],
    dtype="string",
    chunksize=500_000
):
    display_codes.update(chunk["display"].unique())
    mailer_codes.update(chunk["mailer"].unique())

print(f"Display codes: {sorted(display_codes)}")
print(f"Mailer codes: {sorted(mailer_codes)}")

Display codes: ['0', '1', '2', '3', '4', '5', '6', '7', '9', 'A']
Mailer codes: ['0', 'A', 'C', 'D', 'F', 'H', 'J', 'L', 'P', 'X', 'Z']


In [99]:
first_promotion_week = None
last_promotion_week = None

for chunk in pd.read_csv(
    raw_data_path / "causal_data.csv",
    usecols=["WEEK_NO"],
    chunksize=500_000
):
    chunk_min = chunk["WEEK_NO"].min()
    chunk_max = chunk["WEEK_NO"].max()

    if first_promotion_week is None:
        first_promotion_week = chunk_min
        last_promotion_week = chunk_max
    else:
        first_promotion_week = min(first_promotion_week, chunk_min)
        last_promotion_week = max(last_promotion_week, chunk_max)

print(f"First promotion week: {first_promotion_week}")
print(f"Last promotion week: {last_promotion_week}")

First promotion week: 9
Last promotion week: 101


In [100]:
catalog_product_ids = set(product_clean["PRODUCT_ID"])
unknown_promotion_products = set()

for chunk in pd.read_csv(
    raw_data_path / "causal_data.csv",
    usecols=["PRODUCT_ID"],
    chunksize=500_000
):
    unknown_ids = chunk.loc[
        ~chunk["PRODUCT_ID"].isin(catalog_product_ids),
        "PRODUCT_ID"
    ]

    unknown_promotion_products.update(unknown_ids.unique())

print(
    f"Distinct promotion product IDs missing from the catalog: "
    f"{len(unknown_promotion_products)}"
)

Distinct promotion product IDs missing from the catalog: 0


In [101]:
import duckdb

promotion_key_check = duckdb.sql(
    """
    SELECT COUNT(*) AS repeated_product_store_week_keys
    FROM (
        SELECT PRODUCT_ID, STORE_ID, WEEK_NO
        FROM read_csv(?, all_varchar=true)
        GROUP BY PRODUCT_ID, STORE_ID, WEEK_NO
        HAVING COUNT(*) > 1
    )
    """,
    params=[str(raw_data_path / "causal_data.csv")]
).df()

promotion_key_check

,repeated_product_store_week_keys
0,15245


In [102]:
promotion_duplicate_check = duckdb.sql(
    """
    WITH distinct_records AS (
        SELECT DISTINCT
            PRODUCT_ID, STORE_ID, WEEK_NO, display, mailer
        FROM read_csv(?, all_varchar=true)
    ),
    code_variants AS (
        SELECT PRODUCT_ID, STORE_ID, WEEK_NO
        FROM distinct_records
        GROUP BY PRODUCT_ID, STORE_ID, WEEK_NO
        HAVING COUNT(*) > 1
    )
    SELECT COUNT(*) AS keys_with_multiple_code_combinations
    FROM code_variants
    """,
    params=[str(raw_data_path / "causal_data.csv")]
).df()

promotion_duplicate_check

,keys_with_multiple_code_combinations
0,15245


In [103]:
promotion_example = duckdb.sql(
    """
    WITH promotions AS (
        SELECT *
        FROM read_csv(?, all_varchar=true)
    ),
    repeated_key AS (
        SELECT PRODUCT_ID, STORE_ID, WEEK_NO
        FROM promotions
        GROUP BY PRODUCT_ID, STORE_ID, WEEK_NO
        HAVING COUNT(*) > 1
        ORDER BY PRODUCT_ID, STORE_ID, WEEK_NO
        LIMIT 1
    )
    SELECT DISTINCT p.*
    FROM promotions AS p
    JOIN repeated_key AS k
        USING (PRODUCT_ID, STORE_ID, WEEK_NO)
    ORDER BY display, mailer
    """,
    params=[str(raw_data_path / "causal_data.csv")]
).df()

promotion_example

,PRODUCT_ID,STORE_ID,WEEK_NO,display,mailer
0,1001033,323,44,0,A
1,1001033,323,44,1,A


In [104]:
promotion_summary_example = duckdb.sql(
    """
    SELECT
        PRODUCT_ID,
        STORE_ID,
        WEEK_NO,
        string_agg(DISTINCT display, ', ' ORDER BY display) AS display_codes,
        string_agg(DISTINCT mailer, ', ' ORDER BY mailer) AS mailer_codes,
        bool_or(display <> '0') AS any_display_recorded,
        (
            bool_or(display = '0')
            AND bool_or(display <> '0')
        ) AS display_conflict
    FROM read_csv(?, all_varchar=true)
    WHERE PRODUCT_ID = '1001033'
      AND STORE_ID = '323'
      AND WEEK_NO = '44'
    GROUP BY PRODUCT_ID, STORE_ID, WEEK_NO
    """,
    params=[str(raw_data_path / "causal_data.csv")]
).df()

promotion_summary_example

,PRODUCT_ID,STORE_ID,WEEK_NO,display_codes,mailer_codes,any_display_recorded,display_conflict
0,1001033,323,44,"0, 1",A,True,True


In [105]:
processed_data_path = project_root / "data" / "processed"

processed_data_path.mkdir(parents=True, exist_ok=True)

print(processed_data_path)

/Users/faridoonfarahi/Documents/Personal_Projects/customer_loyalty_platform/data/processed


In [106]:
promotion_summary_path = processed_data_path / "promotion_summary.parquet"

duckdb.sql(
    """
    SELECT
        CAST(PRODUCT_ID AS BIGINT) AS PRODUCT_ID,
        CAST(STORE_ID AS BIGINT) AS STORE_ID,
        CAST(WEEK_NO AS BIGINT) AS WEEK_NO,
        string_agg(DISTINCT display, ', ' ORDER BY display) AS display_codes,
        string_agg(DISTINCT mailer, ', ' ORDER BY mailer) AS mailer_codes,
        bool_or(display <> '0') AS any_display_recorded,
        (
            bool_or(display = '0')
            AND bool_or(display <> '0')
        ) AS display_conflict
    FROM read_csv(?, all_varchar=true)
    GROUP BY PRODUCT_ID, STORE_ID, WEEK_NO
    """,
    params=[str(raw_data_path / "causal_data.csv")]
).write_parquet(str(promotion_summary_path), compression="zstd")

print(f"Saved: {promotion_summary_path}")

Saved: /Users/faridoonfarahi/Documents/Personal_Projects/customer_loyalty_platform/data/processed/promotion_summary.parquet


In [107]:
promotion_summary_check = duckdb.sql(
    """
    SELECT
        COUNT(*) AS summary_rows,
        COUNT(*) FILTER (
            WHERE display_conflict
        ) AS display_conflict_rows
    FROM read_parquet(?)
    """,
    params=[str(promotion_summary_path)]
).df()

promotion_summary_check

,summary_rows,display_conflict_rows
0,36771279,15208


In [108]:
promotion_uniqueness_check = duckdb.sql(
    """
    SELECT COUNT(*) AS repeated_keys
    FROM (
        SELECT PRODUCT_ID, STORE_ID, WEEK_NO
        FROM read_parquet(?)
        GROUP BY PRODUCT_ID, STORE_ID, WEEK_NO
        HAVING COUNT(*) > 1
    )
    """,
    params=[str(promotion_summary_path)]
).df()

promotion_uniqueness_check

,repeated_keys
0,0


In [109]:
product_clean_path = processed_data_path / "product_clean.parquet"

product_clean.to_parquet(product_clean_path, index=False)

print(f"Saved {len(product_clean):,} products to: {product_clean_path}")

Saved 92,353 products to: /Users/faridoonfarahi/Documents/Personal_Projects/customer_loyalty_platform/data/processed/product_clean.parquet


In [110]:
coupon_clean_path = processed_data_path / "coupon_clean.parquet"

coupon_clean.to_parquet(coupon_clean_path, index=False)

print(f"Saved {len(coupon_clean):,} coupon eligibility records to: {coupon_clean_path}")

Saved 119,384 coupon eligibility records to: /Users/faridoonfarahi/Documents/Personal_Projects/customer_loyalty_platform/data/processed/coupon_clean.parquet


In [111]:
campaign_summary_path = processed_data_path / "campaign_summary.parquet"

campaign_summary.to_parquet(campaign_summary_path, index=False)

print(f"Saved {len(campaign_summary):,} campaigns to: {campaign_summary_path}")

Saved 30 campaigns to: /Users/faridoonfarahi/Documents/Personal_Projects/customer_loyalty_platform/data/processed/campaign_summary.parquet


In [112]:
transaction_data_path = processed_data_path / "transaction_data.parquet"

transaction_data.to_parquet(
    transaction_data_path,
    index=False,
    compression="zstd"
)

print(f"Saved {len(transaction_data):,} transaction rows to: {transaction_data_path}")

Saved 2,595,732 transaction rows to: /Users/faridoonfarahi/Documents/Personal_Projects/customer_loyalty_platform/data/processed/transaction_data.parquet


In [113]:
demographic_path = processed_data_path / "hh_demographic.parquet"

hh_demographic.to_parquet(demographic_path, index=False)

print(f"Saved {len(hh_demographic):,} demographic records to: {demographic_path}")

Saved 801 demographic records to: /Users/faridoonfarahi/Documents/Personal_Projects/customer_loyalty_platform/data/processed/hh_demographic.parquet


In [114]:
campaign_table_path = processed_data_path / "campaign_table.parquet"

campaign_table.to_parquet(campaign_table_path, index=False)

print(f"Saved {len(campaign_table):,} targeting records to: {campaign_table_path}")

Saved 7,208 targeting records to: /Users/faridoonfarahi/Documents/Personal_Projects/customer_loyalty_platform/data/processed/campaign_table.parquet


In [115]:
coupon_redempt_path = processed_data_path / "coupon_redempt.parquet"

coupon_redempt.to_parquet(coupon_redempt_path, index=False)

print(f"Saved {len(coupon_redempt):,} redemption records to: {coupon_redempt_path}")

Saved 2,318 redemption records to: /Users/faridoonfarahi/Documents/Personal_Projects/customer_loyalty_platform/data/processed/coupon_redempt.parquet


In [116]:
campaign_desc_path = processed_data_path / "campaign_desc.parquet"

campaign_desc.to_parquet(campaign_desc_path, index=False)

print(f"Saved {len(campaign_desc):,} campaign definitions to: {campaign_desc_path}")

Saved 30 campaign definitions to: /Users/faridoonfarahi/Documents/Personal_Projects/customer_loyalty_platform/data/processed/campaign_desc.parquet


In [117]:
import pyarrow.parquet as pq

for file in sorted(processed_data_path.glob("*.parquet")):
    row_count = pq.read_metadata(file).num_rows
    print(f"{file.name}: {row_count:,} rows")

campaign_desc.parquet: 30 rows
campaign_summary.parquet: 30 rows
campaign_table.parquet: 7,208 rows
coupon_clean.parquet: 119,384 rows
coupon_redempt.parquet: 2,318 rows
hh_demographic.parquet: 801 rows
product_clean.parquet: 92,353 rows
promotion_summary.parquet: 36,771,279 rows
transaction_data.parquet: 2,595,732 rows
